In [1]:
# Instalamos las librerías necesarias
!pip install -q -U transformers accelerate bitsandbytes

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 127.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00


In [2]:
# 1. Configuración de cuantización (4-bits)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Nombre del modelo (usaremos la versión instruct para que siga órdenes)
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# 3. Cargamos el Tokenizer y el Modelo
# Considerar el token creado en Hugging Face.
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config
    # Eliminamos device_map="auto" ya que bitsandbytes lo maneja automáticamente para modelos cuantizados
)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [3]:
# INFERENCIA
def preguntar_a_mistral(pregunta, contexto_mensajes=None):
    # Si no se proporciona contexto, inicializamos con la pregunta del usuario
    if contexto_mensajes is None:
        mensajes = [
            {"role": "user", "content": pregunta}
        ]
    else:
        # Si se proporciona contexto, lo usamos y añadimos la pregunta actual
        mensajes = contexto_mensajes + [{"role": "user", "content": pregunta}]

    # Preparamos el input
    encod_input = tokenizer.apply_chat_template(mensajes, return_tensors="pt").to("cuda")

    # Generamos la respuesta
    output = model.generate(**encod_input, max_new_tokens=200, do_sample=True, temperature=0.7)

    # Decodificamos y extraemos solo la respuesta del asistente
    respuesta_completa = tokenizer.decode(output[0], skip_special_tokens=False)

    # Extraemos solo la parte de la respuesta del asistente
    # El formato del chat template para Mistral es algo como: <s>[INST] pregunta [/INST] respuesta </s>
    # Para un historial: <s>[INST] ctx [/INST] resp </s> [INST] pregunta [/INST] respuesta </s>

    # Buscamos el último [/INST] para encontrar el inicio de la última respuesta del asistente.
    # Si hay '</s>' al final, lo quitamos para facilitar la búsqueda del final de la respuesta.
    respuesta_limpia = respuesta_completa.replace("</s>", "").strip()

    # Buscamos el último '["content": ]' o similar para extraer la última respuesta del modelo
    # La forma más robusta es re-tokenizar y buscar el último mensaje del asistente

    # Una aproximación simple: buscar la última aparición de '[/INST]' y obtener lo que sigue
    parts = respuesta_limpia.split('[/INST]')
    if len(parts) > 1:
        # La última parte contiene la respuesta del asistente (puede haber más texto si el modelo sigue)
        # Tomamos la última parte y eliminamos cualquier texto que parezca una nueva instrucción del usuario
        last_part = parts[-1].strip()
        # A veces el modelo genera un nuevo [INST] si llega al límite de tokens o si está "alucinando"
        final_respuesta = last_part.split('[INST]')[0].strip()
        return final_respuesta
    else:
        # Si no se encuentra [/INST], algo fue mal o es una respuesta muy corta sin el formato esperado.
        return respuesta_completa # Devolvemos la respuesta completa como fallback

# Ejemplo de uso (el ejemplo anterior funcionará igual, pero ahora puedes pasar contexto)
print("Ejemplo sin contexto:")
print(preguntar_a_mistral("Explica la mecánica de rocas como si fuera para un niño."))

print("\nEjemplo con contexto:")
mis_mensajes = [
    {"role": "system", "content": "Eres un experto en geología que enseña a niños de 5 años."}, # Un mensaje de sistema para definir el rol
    {"role": "user", "content": "¿Qué es la Tierra?"}, # Un turno anterior del usuario
    {"role": "assistant", "content": "La Tierra es como una gran pelota gigante donde vivimos. Tiene montañas, océanos y muchos animales."}, # Un turno anterior del asistente
]

# Ahora hacemos una nueva pregunta, usando el historial y el mensaje de sistema como contexto
print(preguntar_a_mistral("¿Y qué hay dentro de ella?", contexto_mensajes=mis_mensajes))


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Ejemplo sin contexto:


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


¡Hola! Voy a explicar la mecánica de rocas de una manera sencilla y divertida. Imagina que los cristales y fragmentos de diferentes tipos de rocas son como los juguetes de un niño.

Cada uno de estos juguetes tiene diferentes formas, colores y tamaños, justo como cada tipo de roca tiene una composición, un color y una textura diferente. Algunos de estos juguetes (rocas) pueden ser duros y duraderos, mientras que otros pueden ser más delicados.

Ahora, al igual que cuando un niño juega con sus juguetes, también pasa con las rocas: puede haber fuerzas que las presionan, las rompen, las agitan o las conectan. Esto se conoce como la mec

Ejemplo con contexto:
La Tierra tiene tres partes principales: la corteza, la mantle y el núcleo.

La corteza es la parte más fina y es la que podemos ver y caminar sobre. En ella hay montañas, volcanes y lagos.

La mantle es la parte intermedia, es mucho más caliente que la corteza y está compuesta por rocas fluidas.

El núcleo es la parte más caliente y 